In [1]:
#Import necessary packages:
import rasterio as rio 
import xarray as xr
import rio_cogeo.cogeo
import matplotlib.pyplot as plt
from rasterio.plot import show
import geopandas as gpd
from shapely.geometry import Polygon
import glob

In [ ]:
input_folder ='sst/input'
output_folder='sst/output'

In [ ]:
# Define AOI as a shapely Polygon
aoi_coords = [[-102.8148701375, 6.1943456775], [-13.3448605043, 6.1943456775], 
              [-13.3448605043, 49.6429910636], [-102.8148701375, 49.6429910636], 
              [-102.8148701375, 6.1943456775]]
aoi_polygon = Polygon(aoi_coords)
aoi_gdf = gpd.GeoDataFrame({'geometry': [aoi_polygon]}, crs="EPSG:4326")

In [ ]:
for file in glob.glob("sst/input/*.nc"):

    xds = xr.open_dataset(file)
    data_array = xds['SST'].values[::-1,:]
    lat_array = xds.lat.values
    lon_array = xds.lon.values
    data_array.shape, lat_array.shape, lon_array.shape
    
    # Create the DataArray
    data_xarray = xr.DataArray(
        data_array,
        coords={
            'lat': lat_array,
            'lon': lon_array
        },
        dims=[ 'lat', 'lon']
    )

    data_xarray.rio.set_spatial_dims("lon", "lat", inplace=True)
    data_xarray.rio.write_crs("epsg:4326", inplace=True)

    data_xarray = data_xarray.rio.clip(aoi_gdf.geometry, aoi_gdf.crs, drop=True)
    
    # Set nodata value using set_nodata
    data_xarray = data_xarray.where(data_xarray != 0, -9999)
    data_xarray = data_xarray.rio.set_nodata(-9999)

    date = file.split("/")[-1].split("_")[0]
    formatted_datetime = f"{date[:4]}-{date[4:6]}-{date[6:]}T00:00:00Z"

    # Write the COG
    data_xarray.rio.to_raster(
        f"{output_folder}/SST_{formatted_datetime}.tif",
        driver="COG",
        compress="DEFLATE",
        overview_level=4,
        overview_resampling="average",
        nodata=-9999
    )
print("Done transforming")
    